# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Ranking Signal Analysis. It has the most data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The dataset helps the SEO/content team make data‑driven decisions about keyword and content strategy, errors in those decisions translate into measurable traffic, revenue, and resource losses.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# Portable setup — works in Colab and locally, no hardcoded paths.
import os, sys, subprocess
from pathlib import Path
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/si-ux/FlyrankAI-Internship"
REPO_DIR = "FlyrankAI-Internship"
MARKER = "data/raw/content_refresh_anonymized.csv"

if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Walk up from here (or into the fresh clone) until the starter CSV is in sight.
ROOT = Path(REPO_DIR).resolve() if os.path.isdir(REPO_DIR) else Path.cwd()
while not (ROOT / MARKER).exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / MARKER).exists(), "starter CSV not found — run this from inside the repo"

df = pd.read_csv(ROOT / MARKER)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
df.head(10)

Loaded 30,000 rows x 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [4]:
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# ── 1. Scale of declining content ────────────────────────────────────────────
n_declining = df["is_declining"].sum()
pct_declining = df["is_declining"].mean() * 100
print(f"[1] Declining pages: {n_declining:,} / {len(df):,}  ({pct_declining:.1f}% of portfolio)")

# ── 2. Impressions being wasted on declining pages ────────────────────────────
imp_declining  = df.loc[df["is_declining"]==1, "impressions_90d"].sum()
imp_total      = df["impressions_90d"].sum()
print(f"\n[2] Impressions (90d) on declining pages: {imp_declining:,.0f}"
      f"  ({imp_declining/imp_total*100:.1f}% of all impressions)")

# ── 3. CTR gap — declining pages convert impressions to clicks far worse ─────
ctr_by_trend = df.groupby("trend_direction")["ctr"].median()
print(f"\n[3] Median CTR — declining: {ctr_by_trend.get('down', 0):.2f}%"
      f"  |  improving: {ctr_by_trend.get('up', 0):.2f}%")

# ── 4. High-value keywords being left to decline (CPC × search_volume proxy) ──
df["traffic_value_proxy"] = df["cpc"] * df["clicks_90d"]
high_value_declining = df[(df["is_declining"]==1) & (df["cpc"] >= df["cpc"].quantile(0.75))]
print(f"\n[4] High-CPC pages that are declining: {len(high_value_declining):,}"
      f"  |  Est. value at risk: ${high_value_declining['traffic_value_proxy'].sum():,.0f}")

# ── 5. Stale content dominates the declining segment ─────────────────────────
stale_threshold = 180  # days since last update
stale_declining = df[(df["is_declining"]==1) & (df["days_since_last_update"] >= stale_threshold)]
print(f"\n[5] Declining pages not updated in 6+ months: {len(stale_declining):,}"
      f"  ({len(stale_declining)/n_declining*100:.1f}% of all declining pages)")

# ── 6. Position decay — declining pages sit further from page 1 ───────────────
pos_by_trend = df.groupby("trend_direction")["avg_position"].median()
print(f"\n[6] Median avg position — declining: {pos_by_trend.get('down', 0):.1f}"
      f"  |  improving: {pos_by_trend.get('up', 0):.1f}  (higher = worse rank)")

# ── 7. Resource misallocation — long content with zero engagement declining ───
wasted_effort = df[
    (df["is_declining"]==1) &
    (df["word_count"] >= 2000) &
    (df["engaged_sessions_90d"] == 0)
]
print(f"\n[7] Long (2000+ word) declining pages with zero engaged sessions: {len(wasted_effort):,}"
      f"  — production cost with no return")

# ── 8. Striking distance pages declining — missed ranking opportunities ────────
striking_declining = df[(df["position_tier"] == "striking") & (df["is_declining"]==1)]
print(f"\n[8] 'Striking distance' pages (pos 4–20) that are declining: {len(striking_declining):,}"
      f"  — close to page 1 but losing ground")

[1] Declining pages: 16,262 / 30,000  (54.2% of portfolio)

[2] Impressions (90d) on declining pages: 79,994,363  (51.3% of all impressions)

[3] Median CTR — declining: 0.08%  |  improving: 0.09%

[4] High-CPC pages that are declining: 15,525  |  Est. value at risk: $54,213

[5] Declining pages not updated in 6+ months: 82  (0.5% of all declining pages)

[6] Median avg position — declining: 11.3  |  improving: 15.3  (higher = worse rank)

[7] Long (2000+ word) declining pages with zero engaged sessions: 7,254  — production cost with no return

[8] 'Striking distance' pages (pos 4–20) that are declining: 4,452  — close to page 1 but losing ground


More than half the portfolio — **16,262 of 30,000 pages (54.2%)** — are declining in traffic. These pages account for **51.3% of all impressions** over the past 90 days, meaning the majority of the site's search visibility is being absorbed by content that is actively losing ground. That represents a structural inefficiency: the audience is finding these pages, but the pages are failing to hold their position or convert.

The conversion gap reinforces this. Declining pages have a median CTR of **0.08%** versus **0.09%** for improving pages — a small gap per page, but multiplied across millions of impressions, it compounds into significant lost clicks and traffic.

The risk is sharpest at the high-value end. **15,525 pages with above-average CPC are declining**, representing an estimated **\$54,213 in traffic value at risk** — real ad-equivalent revenue the site is failing to capture because these pages are slipping in rank, not because the keywords are unworthy.

Position data confirms the structural decay: declining pages sit at a median average position of **11.3**, compared to **15.3** for improving pages — meaning declining pages are paradoxically *closer* to page 1 but still losing ground, likely due to missed refresh cycles. Supporting this, **4,452 pages in striking distance (positions 4–20) are declining** — these are the highest-opportunity pages where a small intervention could push them onto page 1, but without action they're drifting the wrong way.

The resource waste angle is equally stark. **7,254 pages of 2,000+ words have zero engaged sessions** in 90 days while declining — production spend with no measurable return. Yet only **82 declining pages (0.5%)** were updated in the last 6 months, meaning the team is not systematically targeting the problem.

Taken together, the data shows that without a principled method to triage which pages to refresh, prioritize, or retire, the team is effectively blind to where traffic, rankings, and production budgets are leaking.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

The work delivers descriptive and predictive‑directional insights that help prioritize SEO and content decisions, but it does not deliver causal guarantees or a crystal‑ball view of Google’s future ranking behavior.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.